# บทเรียน 02 - การสำรวจ Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** คือกรอบงานแบบรวมสำหรับการสร้างเอไอเอเจนต์ มันมีสถาปัตยกรรมที่สะอาดและประกอบได้ที่มีบล็อกตัวสร้างหลักสี่บล็อก:

- **Client** – เชื่อมต่อกับปลายทางโมเดลเอไอและจัดการการสื่อสาร
- **Agent** – ห่อหุ้มไคลเอนต์ด้วยคำสั่งและนิยามเครื่องมือ
- **Tools** – ขยายความสามารถของเอเจนต์ด้วยฟังก์ชันที่กำหนดเองซึ่งโมเดลสามารถเรียกใช้ได้
- **Session** – เก็บประวัติการสนทนาสำหรับการโต้ตอบหลายรอบ

ในบทเรียนนี้ เราจะสร้าง **เอเจนต์จองท่องเที่ยว** ที่ตรวจสอบความพร้อมของปลายทางโดยใช้แนวคิดเหล่านี้


## การตั้งค่า


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## การเข้าใจสถาปัตยกรรมของ Agent Framework

Microsoft Agent Framework ใช้สถาปัตยกรรมแบบมีชั้นดังนี้:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` เชื่อมต่อกับการปรับใช้ Azure OpenAI มันจัดการการตรวจสอบสิทธิ์ การจัดรูปแบบคำขอ และการแยกวิเคราะห์การตอบกลับ
2. **Agent** – สร้างจากไคลเอนต์ผ่าน `provider.create_agent()` เอเจนต์รวมการเข้าถึงโมเดลเข้ากับคำแนะนำ (system prompt) และเครื่องมือ
3. **Tools** – ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่งเอเจนต์สามารถเรียกใช้เพื่อดำเนินการหรือเรียกข้อมูล
4. **Session** – อ็อบเจ็กต์ `AgentSession` (สร้างผ่าน `agent.create_session()`) ที่เก็บประวัติการสนทนา ทำให้สามารถสนทนาได้หลายรอบโดยที่เอเจนต์จำบริบทก่อนหน้าได้

มาสร้างแต่ละชั้นทีละขั้นตอนกันเถอะ


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การเพิ่มเครื่องมือด้วย @tool Decorator

เครื่องมือช่วยให้เอเย่นต์สามารถดำเนินการได้มากกว่าการสร้างข้อความ `@tool` decorator แปลงฟังก์ชัน Python ธรรมดาให้เป็นสิ่งที่เอเย่นต์สามารถเรียกใช้ได้

ประเด็นสำคัญ:
- ใช้ `Annotated[type, "description"]` เพื่อให้โมเดลเข้าใจพารามิเตอร์แต่ละตัว
- docstring จะกลายเป็นคำอธิบายเครื่องมือที่โมเดลเห็น
- `approval_mode="never_require"` หมายความว่าเครื่องมือทำงานโดยอัตโนมัติโดยไม่ต้องยืนยันจากผู้ใช้


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## การสร้างเอเจนต์ด้วยเครื่องมือ

ตอนนี้เราจะรวมไคลเอนต์ คำแนะนำ และเครื่องมือเข้าด้วยกันเป็นเอเจนต์ โดยที่ `instructions` ทำหน้าที่เป็นพรอมต์ระบบ — กำหนดบุคลิกและพฤติกรรมของเอเจนต์


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## การสนทนาแบบหลายรอบด้วย Sessions

`AgentSession` (สร้างขึ้นผ่าน `agent.create_session()`) จะเก็บประวัติข้อความทั้งหมดในการสนทนา โดยการส่ง session เดียวกันไปยังแต่ละการเรียก `agent.run()` ตัวแทนจะสามารถเข้าถึงประวัติการสนทนาแบบเต็มและสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าได้

เราส่ง `tools=[check_destination_availability]` เพื่อให้ตัวแทนสามารถเรียกใช้ตัวตรวจสอบความพร้อมใช้งานของเราในแต่ละรอบได้


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## สรุป

ในบทเรียนนี้ คุณได้สำรวจเสาหลักสี่ด้านของ Microsoft Agent Framework:

| แนวคิด | สิ่งที่คุณได้เรียนรู้ |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` เชื่อมต่อกับ Azure OpenAI ด้วยการยืนยันตัวตนแบบใช้ข้อมูลรับรอง |
| **Agent** | `provider.create_agent()` รวบรวมการเชื่อมต่อโมเดลพร้อมคำแนะนำและชื่อ |
| **Tools** | ตัวตกแต่ง `@tool` เปิดเผยฟังก์ชัน Python สำหรับให้ตัวแทนเรียกใช้ |
| **Session** | `agent.create_session()` รักษาประวัติการสนทนาในหลายรอบ |

บล็อกการสร้างเหล่านี้ประกอบเข้าด้วยกันเพื่อสร้างตัวแทนที่สามารถสนทนาแบบธรรมชาติ, เรียกใช้ฟังก์ชันภายนอก และรักษาบริบท — ซึ่งเป็นรากฐานสำหรับรูปแบบตัวแทนที่ซับซ้อนกว่านี้ในบทเรียนต่อไป.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้ความถูกต้องสูงสุด โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อนได้ เอกสารต้นฉบับในภาษาต้นฉบับควรถูกพิจารณาเป็นแหล่งข้อมูลที่ถูกต้อง สำหรับข้อมูลสำคัญ แนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญ เพื่อหลีกเลี่ยงความเข้าใจผิดหรือการแปลความหมายที่ผิดพลาด เราจะไม่รับผิดชอบใดๆ ที่เกิดขึ้นจากการใช้การแปลฉบับนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
